# Vehicle Type Classification — Final Consolidated Machine Learning Project

**Course-submission notebook | Final five Kaggle candidates | 14 October 2026**

This is a reproducible consolidation of the originally uploaded `Vehicle_Classification.ipynb` and subsequent experiment notebooks through **Round 8**. It focuses on the methods in the taught syllabus: ridge/lasso least squares, Gaussian generative models, LDA/QDA/Naïve Bayes, binary/multinomial logistic regression, polynomial/interaction features, and decision-tree methods. **Elastic Net and ensembles are permitted according to the course clarification.** Non-syllabus algorithms screened earlier (e.g., SVM and boosting packages) are noted as historical screening only, and are not used in the final five.

**Confirmed best public score: 0.90500.** Public accuracy is not a proxy for private accuracy; the private labels are unavailable. Five finalists are selected to preserve strong public score **and** model diversity.

| Final priority | Original Kaggle submission CSV | Confirmed public accuracy | Model |
|---:|---|---:|---|
| 1 | `round4_blend_baseline_elastic.csv` | **0.90500** | 50% drop-city multinomial logistic + 50% full-feature Elastic Net logistic |
| 2 | `vehicle_submission_drop_city.csv` | **0.90400** | Multinomial logistic, 19 predictors, L2 C=1 |
| 3 | `round3_elastic_full_5.csv` | **0.90400** | Multinomial logistic, 20 predictors, Elastic Net C=5, L1 ratio=0.5 |
| 4 | `vehicle_submission_engineered.csv` | **0.90300** | Multinomial logistic, 20 numeric predictors + 8 engineered features |
| 5 | `round5_lda20.csv` | **0.90200** | 40% drop-city logistic + 40% Elastic Net + 20% Gaussian LDA |

**Why these five?** The ensemble holds the best observed public score; two components provide straightforward and independently fitted alternatives; the engineered model explores feature-based diversity; and the LDA-assisted blend adds a distinct Gaussian generative classification assumption. A standalone L2 baseline at **0.90200** is retained as a reserve candidate. The last two aren't predictions of superior private performance—private accuracy is unknown.

**Environment:** Colab, Kaggle, or a local Jupyter kernel. Put `vehicle_train_10000.csv`, `vehicle_test.csv` and `vehicle_sample_submission.csv` in a reachable directory. The core notebook generates five corresponding CSVs (prefixed `top0N_`) and records fold-appropriate evaluation. Some historical diagnostic sections are optional and disabled by default to save compute.

## 1. Experiment provenance and scope

This notebook reproduces the *methodological content* of the initial notebook while removing its Git configuration and a hardcoded repository credential. The original contained a misspelled parameter `C=t0.1`, corrected here to `C=0.1` for its softmax candidate. No credentials or remote repository operations are required.

**Sources consolidated:** the initial notebook; `Vehicle_Classification_91plus_Experiments.ipynb`; `Vehicle_Classification_Data_Optimized.ipynb`; and the notebooks from Rounds 2–8, along with validation-result CSVs and user-confirmed Kaggle submissions. The archive includes a sanitized original notebook and original source experiment notebooks.

**Metric interpretation:** CV means across shuffled folds reuse the same 7,500 unique training records, so they are not independent datasets and must not be treated as confidence intervals. Kaggle scores listed above were reported after actual submissions. The public/private partition sizes and private test labels are unknown.

In [ ]:
import os, warnings, math
for _v in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'): os.environ.setdefault(_v,'1')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, log_loss, confusion_matrix, classification_report
warnings.filterwarnings('ignore', category=FutureWarning)
SEED=42
# Changing these flags will add experiments; core finalists always train and export.
RUN_EXTENDED_BENCHMARK=False
RUN_EXTRA_DIAGNOSTICS=False
RUN_REPEATED_FINALIST_CV=True
print('Python setup ready')

## 2. Dataset loading, features, target, duplicates, and missingness

The supplied training file has 10,000 rows, a `vehicle_id` identifier, **20 numeric predictor columns**, and an encoded target `vehicle_class` ∈ {0,1,2}. The competition test file has 2,500 rows and no labels. Exactly 2,500 training rows repeat the features of an earlier row with the same label, leaving 7,500 distinct labeled feature vectors. Deduplication is performed **before** fold creation to avoid optimistic CV results from repeated examples. There are no missing values or train–test exact feature duplicates in the provided CSVs.

The numeric labels are retained as class 0/1/2; no unsupported semantic class names are invented.

In [ ]:
FILES=['vehicle_train_10000.csv','vehicle_test.csv','vehicle_sample_submission.csv']
DATA_DIR=None  # Optional: Path to dataset; autodetect Kaggle/Colab/current working directory.
def locate(fn):
    if DATA_DIR is not None and (Path(DATA_DIR)/fn).is_file(): return Path(DATA_DIR)/fn
    for root in [Path('/kaggle/input'),Path('/mnt/data'),Path('/content'),Path.cwd()]:
        direct=root/fn
        if direct.is_file():return direct
        if root.exists():
            paths=list(root.rglob(fn))
            if paths:return paths[0]
    raise FileNotFoundError(f'Attach {fn} using Kaggle Add Input or Colab Files, or set DATA_DIR')
files={fn:locate(fn) for fn in FILES}
train=pd.read_csv(files[FILES[0]]);test=pd.read_csv(files[FILES[1]]);sample=pd.read_csv(files[FILES[2]])
ID='vehicle_id';TARGET='vehicle_class';FEATURES=[c for c in train if c not in (ID,TARGET)]
assert len(FEATURES)==20 and set(FEATURES).issubset(test.columns)
assert sample[ID].equals(test[ID]),'Never reorder test IDs.'
conflicts=train.groupby(FEATURES,dropna=False)[TARGET].nunique().gt(1).sum()
assert conflicts==0,'Conflicting labels for identical feature vectors.'
unique=train.drop_duplicates(subset=FEATURES).reset_index(drop=True)
y=unique[TARGET].to_numpy(); classes=np.sort(np.unique(y))
assert list(classes)==[0,1,2]
DROP_CITY=[c for c in FEATURES if c!='city_fuel_efficiency_kmpl']
print('Train / test / unique:',train.shape,test.shape,unique.shape)
print('Duplicate feature vectors:',len(train)-len(unique),'| Label conflicts:',conflicts)
print('Missing train/test:',train[FEATURES].isna().sum().sum(),test[FEATURES].isna().sum().sum())
print('Unique class distribution:',unique[TARGET].value_counts().sort_index().to_dict())
print('Features:',FEATURES)

## 3. Exploratory Data Analysis (EDA)

Inspect class frequencies, feature correlations and distributions. All plots use training features only. High collinearity (for example, variables linked to vehicle size, engine power and torque) motivates regularization and makes complex polynomial expansions susceptible to overfitting.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,3.6))
unique[TARGET].value_counts().sort_index().plot(kind='bar',ax=ax[0],title='Unique training rows by class')
ax[0].set(xlabel='Class',ylabel='Count');
for cl in sorted(classes):
    ax[1].hist(unique.loc[unique[TARGET]==cl,'horsepower_hp'],bins=22,alpha=.45,label=f'class {cl}')
ax[1].set(xlabel='Horsepower (hp)',ylabel='Vehicles',title='Horsepower distributions');ax[1].legend()
plt.tight_layout();plt.show()
cor=unique[FEATURES].corr(numeric_only=True)
plt.figure(figsize=(8.5,6));plt.imshow(cor,vmin=-1,vmax=1,cmap='coolwarm');plt.colorbar(shrink=.75)
plt.xticks(range(len(FEATURES)),FEATURES,rotation=90,fontsize=7);plt.yticks(range(len(FEATURES)),FEATURES,fontsize=7)
plt.title('Numeric feature correlations');plt.tight_layout();plt.show()

## 4. Preprocessing pipeline and `ColumnTransformer`

**Handling missing values:** none were found, but a fold-fitted median imputer is shown as a defensive option. **Categorical encoding:** no predictors are categorical, so `OneHotEncoder` is not applied. **Scaling:** `StandardScaler` is needed for penalized logistic and least-squares models; the scaler must be fitted only on the training fold.

Below is the teaching-focused `ColumnTransformer` pipeline. **For exact reproduction of the historical finalist CSVs**, later cells use `StandardScaler` directly, which is equivalent on this fully observed numeric-only dataset and matches the original fitted pipeline.

In [ ]:
numeric_features=list(FEATURES)
categorical_features=[]  # confirmed: none in the supplied competition dataset
transformers=[('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric_features)]
if categorical_features:
    transformers.append(('category',OneHotEncoder(handle_unknown='ignore'),categorical_features))
preprocessor=ColumnTransformer(transformers, remainder='drop')
demo_pipeline=Pipeline([('preprocessing',preprocessor),('classifier',LogisticRegression(C=.1,max_iter=2000))])
# Demonstration only: do not fit on all data before cross-validation.
print(demo_pipeline)

## 5. Feature engineering: log, ratio, interaction, and class-0 features

The early notebooks used an eight-feature block (power/weight, torque/weight, power/engine displacement, city/highway efficiency, wheelbase/length, height/width, weight/footprint and acceleration×power). Later experiments added power/geometry ratios, selected quadratic/cubic interactions and size/volume features. This consolidated notebook additionally *demonstrates* log transforms for completeness; no original Kaggle submission specifically validated their effect. All transformations below are label-free. Feature selection based on CV is never performed on a test-label outcome.

In [ ]:
def engineered_eight(frame):
    z=frame[FEATURES].copy()
    z['power_weight']=1000*z.horsepower_hp/z.vehicle_weight_kg
    z['torque_weight']=1000*z.torque_nm/z.vehicle_weight_kg
    z['hp_liter']=z.horsepower_hp/z.engine_displacement_l
    z['eff_ratio']=z.city_fuel_efficiency_kmpl/z.highway_fuel_efficiency_kmpl
    z['wheel_length']=z.wheelbase_mm/z.vehicle_length_mm
    z['height_width']=z.vehicle_height_mm/z.vehicle_width_mm
    z['weight_area']=1e6*z.vehicle_weight_kg/(z.vehicle_length_mm*z.vehicle_width_mm)
    z['acc_power']=z.acceleration_0_100_kmh_s*z.horsepower_hp
    return z

def exploratory_features(frame,group='physical'):
    z=frame[DROP_CITY].copy();eps=1e-9
    d={
      'hp_per_ton':1000*frame.horsepower_hp/(frame.vehicle_weight_kg+eps),
      'torque_per_ton':1000*frame.torque_nm/(frame.vehicle_weight_kg+eps),
      'wheelbase_to_length':frame.wheelbase_mm/(frame.vehicle_length_mm+eps),
      'width_to_height':frame.vehicle_width_mm/(frame.vehicle_height_mm+eps),
      'body_volume_m3':frame.vehicle_length_mm*frame.vehicle_width_mm*frame.vehicle_height_mm/1e9,
      'eff_gap':frame.highway_fuel_efficiency_kmpl-frame.city_fuel_efficiency_kmpl,
      'log_hp':np.log1p(np.maximum(frame.horsepower_hp,0)),
      'log_displacement':np.log1p(np.maximum(frame.engine_displacement_l,0)),
      'log_weight':np.log1p(np.maximum(frame.vehicle_weight_kg,0))}
    if group in ('physical','physical_poly'):z=z.assign(**d)
    if group in ('focused_poly','physical_poly'):
        scale={'horsepower_hp':(290,100),'vehicle_weight_kg':(1650,450),
               'tire_width_mm':(245,40),'vehicle_width_mm':(1840,220)}
        v={k:(frame[k]-mu)/sigma for k,(mu,sigma) in scale.items()}
        for i,k in enumerate(v):
            for j,l in enumerate(v):
                if j>=i:z[f'{k}__x__{l}']=v[k]*v[l]
    if group=='cubic':
        for k in ['horsepower_hp','vehicle_weight_kg','tire_width_mm']:
            anchors={'horsepower_hp':(290,100),'vehicle_weight_kg':(1650,450),'tire_width_mm':(245,40)}
            mid,scale=anchors[k]
            standardized=(frame[k]-mid)/scale  # fixed domain anchors; no fold-global statistics
            z[f'{k}__cube']=standardized**3
    return z
print('Original / early engineered / focused polynomial feature counts:',len(DROP_CITY),engineered_eight(unique).shape[1],exploratory_features(unique,'focused_poly').shape[1])

## 6. Train/validation strategy and leakage control

The main evaluation uses stratified five-fold **out-of-fold (OOF)** probabilities after deduplication. Finalists are compared using seeds 42, 2026 and 17. These are different partitions of the **same rows** rather than three independent datasets. Earlier rounds also used seeds 777 and 314, a 25% class-stratified holdout and nested feature screening. Pipeline transformers and augmentation noise scales should be fitted within each training fold. The Kaggle public score is used only as a secondary evaluation signal; we cannot assess private labels.

In [ ]:
FEATURE_SETS={
    'baseline':unique[FEATURES],
    'dropcity':unique[DROP_CITY],
    'elastic':unique[FEATURES],
    'engineered':engineered_eight(unique),
}
TEST_SETS={
    'baseline':test[FEATURES],
    'dropcity':test[DROP_CITY],
    'elastic':test[FEATURES],
    'engineered':engineered_eight(test),
}
def model_for(name):
    if name=='baseline':return make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=2000))
    if name=='dropcity':return make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=2000))
    if name=='engineered':return make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=2000))
    if name=='elastic':return make_pipeline(StandardScaler(),LogisticRegression(C=5,l1_ratio=.5,solver='saga',tol=1e-4,max_iter=1800,random_state=777))
    raise ValueError(name)

def lda_oof(seed):
    oof=np.zeros((len(y),3))
    for tr,va in StratifiedKFold(5,shuffle=True,random_state=seed).split(unique,y):
        lda=make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage=0))
        oof[va]=lda.fit(unique[FEATURES].iloc[tr],y[tr]).predict_proba(unique[FEATURES].iloc[va])
    return oof

def make_oof(seed=42,models=('baseline','dropcity','elastic','engineered')):
    P={name:np.zeros((len(y),3)) for name in models}
    for tr,va in StratifiedKFold(5,shuffle=True,random_state=seed).split(unique,y):
        for name in models:
            m=model_for(name).fit(FEATURE_SETS[name].iloc[tr],y[tr])
            P[name][va]=m.predict_proba(FEATURE_SETS[name].iloc[va])
    if {'dropcity','elastic'}.issubset(models):
        P['winner_50_50']=.5*P['dropcity']+.5*P['elastic']
        P['lda20']=.4*P['dropcity']+.4*P['elastic']+.2*lda_oof(seed)
    return P

## 7. Original notebook models and syllabus-compliant model registry

The initial notebook compared **Ridge/Lasso regression on one-hot labels**, Gaussian Naïve Bayes, LDA, regularized QDA and L1/L2 multinomial logistic regression, then explored Gaussian-noise augmentation. Later notebooks explored tree classifiers, Gaussian/shrinkage classifiers, class-specific binary strategies, polynomial logistic models, Elastic Net and probability ensembles. The benchmark below implements representative versions with identical three-fold splits. The `RUN_EXTENDED_BENCHMARK` flag is off by default because the main five finalists are evaluated in Section 8.

In [ ]:
class LeastSquaresClassifier(BaseEstimator,ClassifierMixin):
    def __init__(self,kind='ridge',alpha=1.): self.kind=kind;self.alpha=alpha
    def fit(self,X,y):
        self.classes_=np.unique(y)
        yy=(np.asarray(y)[:,None]==self.classes_).astype(float)
        self.m_=(Ridge(alpha=self.alpha) if self.kind=='ridge' else Lasso(alpha=self.alpha,max_iter=2500)).fit(X,yy)
        return self
    def predict(self,X):return self.classes_[self.m_.predict(X).argmax(axis=1)]

ELIGIBLE_MODELS={
 'Ridge least squares':make_pipeline(StandardScaler(),LeastSquaresClassifier('ridge',10)),
 'Lasso least squares':make_pipeline(StandardScaler(),LeastSquaresClassifier('lasso',.01)),
 'Gaussian Naïve Bayes':make_pipeline(StandardScaler(),GaussianNB()),
 'LDA':make_pipeline(StandardScaler(),LinearDiscriminantAnalysis()),
 'Shrinkage LDA':make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage=.1)),
 'QDA reg=0.1':make_pipeline(StandardScaler(),QuadraticDiscriminantAnalysis(reg_param=.1)),
 'QDA reg=0.3':make_pipeline(StandardScaler(),QuadraticDiscriminantAnalysis(reg_param=.3)),
 'L1 softmax':make_pipeline(StandardScaler(),LogisticRegression(C=.1,l1_ratio=1,solver='saga',max_iter=1800,tol=1e-3,random_state=42)),
 'L2 softmax':model_for('baseline'),
 'Elastic Net softmax':model_for('elastic'),
 'Decision Tree':DecisionTreeClassifier(max_depth=7,min_samples_leaf=10,random_state=42),
 'Random Forest':RandomForestClassifier(n_estimators=100,min_samples_leaf=3,n_jobs=1,random_state=42),
 'Extra Trees':ExtraTreesClassifier(n_estimators=100,min_samples_leaf=3,n_jobs=1,random_state=42),
}
if RUN_EXTENDED_BENCHMARK:
    runs=[]
    for name,m in ELIGIBLE_MODELS.items():
        predicted=np.zeros(len(y),dtype=int)
        for tr,va in StratifiedKFold(3,shuffle=True,random_state=42).split(unique,y):
            m.fit(unique[FEATURES].iloc[tr],y[tr]);predicted[va]=m.predict(unique[FEATURES].iloc[va])
        runs.append({'model':name,'3-fold accuracy':accuracy_score(y,predicted),'macro F1':f1_score(y,predicted,average='macro')})
    print(pd.DataFrame(runs).sort_values('3-fold accuracy',ascending=False).to_string(index=False))
else:
    print('Extended benchmark disabled. Historical results remain documented below.')

### Previously measured eligible-model screening (not directly comparable to finalist five-fold scores)

These values are from a **three-fold seed-42 screening** on the deduplicated data. The final five use separate **five-fold** evaluation, so their numbers should not be interpreted as head-to-head scores without rerunning the same folds.

| Eligible model | Three-fold accuracy |
|---|---:|
| LDA with shrinkage | 89.39% |
| LDA | 89.36% |
| L1 softmax regression | 89.15% |
| QDA with 0.1 covariance regularization | 88.96% |
| Random Forest | 88.17% |
| Extra Trees | 88.15% |
| Ridge least squares | 86.28% |
| Lasso least squares | 86.37% |
| Decision Tree | 86.03% |
| Gaussian Naïve Bayes | 84.68% |

**Historical non-syllabus-only screening:** SVM, CatBoost, XGBoost and LightGBM were explored in earlier model searches but not selected for this course submission; they are intentionally excluded from the executable final registry.

## 8. Cross-validated finalist comparison — accuracy, macro-F1, class-0 recall, log loss

Each finalist is trained from scratch inside each fold. The 50/50 blend averages **probabilities** (not hard labels). CV results may rank models differently from the Kaggle public leaderboard, reinforcing the need for diverse private-test candidates.

In [ ]:
SEEDS=[42,2026,17] if RUN_REPEATED_FINALIST_CV else [42]
cv_rows=[]; conf_winner=None
for sd in SEEDS:
    pp=make_oof(sd)
    for name,p in pp.items():
        pred=classes[p.argmax(1)]
        cv_rows.append({'seed':sd,'model':name,'accuracy':accuracy_score(y,pred),
                        'macro_f1':f1_score(y,pred,average='macro'),
                        'class0_recall':np.mean(pred[y==0]==0),
                        'log_loss':log_loss(y,p,labels=list(classes))})
    if sd==42:conf_winner=confusion_matrix(y,classes[pp['winner_50_50'].argmax(1)],labels=classes)
cv_table=pd.DataFrame(cv_rows)
print('Mean metrics over CV shuffles (same rows reused; not independent repetitions):')
print(cv_table.groupby('model')[['accuracy','macro_f1','class0_recall','log_loss']].mean().round(5).sort_values('accuracy',ascending=False))
print('Winner confusion matrix, seed 42:\n',conf_winner)

## 9. Revisit the original Gaussian-noise augmentation (properly fold-aware)

The original Colab notebook created five Gaussian-perturbed copies with a noise scale proportional to feature standard deviation and compared clean/noisy validation accuracies. A correction is important: **clean validation accuracy** is the primary benchmark for the Kaggle test set; a mixed noisy-validation average is not the competition metric. The version below fits the noise scale using only the training fold (unlike the global scale in the original).

In [ ]:
def fold_noisy_logistic(Xtr,ytr,C=.1,noise=.3,copies=5,seed=42):
    rng=np.random.default_rng(seed)
    sd=np.asarray(Xtr).std(axis=0)
    chunks=[np.asarray(Xtr)] + [np.asarray(Xtr)+rng.normal(size=np.asarray(Xtr).shape)*sd*noise for _ in range(copies)]
    yy=np.tile(np.asarray(ytr),copies+1)
    return make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000)).fit(np.vstack(chunks),yy)
if RUN_EXTRA_DIAGNOSTICS:
    pv=np.zeros((len(y),3))
    for fi,(tr,va) in enumerate(StratifiedKFold(5,shuffle=True,random_state=42).split(unique,y)):
        m=fold_noisy_logistic(unique[FEATURES].iloc[tr],y[tr],seed=42+fi)
        pv[va]=m.predict_proba(unique[FEATURES].iloc[va])
    print('Clean OOF accuracy for augmented model:',accuracy_score(y,pv.argmax(1)))
else:print('Augmentation code preserved; optional diagnostic disabled.')

## 10. Other investigated syllabus-compliant strategies and negative results

Historical rounds: (a) feature ablation (dropping city fuel-efficiency, vehicle-volume features), (b) L1/Elastic Net regularization and stronger L2, (c) LDA/QDA and class-conditional Gaussian assumptions, (d) binary/hierarchical classifiers for class 0 vs nonzero, (e) tree-based models and probability mixing, (f) log/ratio/polynomial/cubic features, (g) class reweighting, (h) pseudo-label tests (unlabeled test features; not kept), (i) class-probability bias/threshold search and (j) exploratory PCA feature transformations. None reliably exceeded the 50/50 blend in repeated training-set CV; the private score remains unknown.

The following functions preserve executable *diagnostic* comparisons. They are not automatically promoted to finalist status, and the threshold grid is diagnostic only: tuning to the Kaggle public labels would constitute leakage/overfitting.

In [ ]:
# Hierarchical classification: class-0 vs nonzero, then class-1 vs class-2.
def hierarchical_model_probs(Xtr,ytr,Xva,C0=1.,C12=1.):
    b0=make_pipeline(StandardScaler(),LogisticRegression(C=C0,max_iter=2000)).fit(Xtr,(ytr!=0).astype(int))
    b12=make_pipeline(StandardScaler(),LogisticRegression(C=C12,max_iter=2000)).fit(Xtr[ytr!=0],ytr[ytr!=0])
    p_nonzero=b0.predict_proba(Xva)[:,1];p12=b12.predict_proba(Xva)
    return np.column_stack([1-p_nonzero,p_nonzero*p12[:,0],p_nonzero*p12[:,1]])

def diagnostic_bias_grid(y_true,oof_proba,values=(-.15,-.05,0,.05,.15)):
    trials=[]
    for b0 in values:
        for b2 in values:
            bias=np.exp(np.array([b0,0,b2])); pred=(oof_proba*bias).argmax(axis=1)
            trials.append((b0,b2,accuracy_score(y_true,pred)))
    return pd.DataFrame(trials,columns=['bias_class0','bias_class2','OOF_accuracy']).sort_values('OOF_accuracy',ascending=False)
if RUN_EXTRA_DIAGNOSTICS:
    h=np.zeros((len(y),3))
    for tr,va in StratifiedKFold(5,shuffle=True,random_state=42).split(unique,y):
        h[va]=hierarchical_model_probs(unique[FEATURES].to_numpy()[tr],y[tr],unique[FEATURES].to_numpy()[va])
    print('Hierarchical OOF accuracy:',accuracy_score(y,h.argmax(1)))
    print('Bias grid top (diagnostic, not nested):',diagnostic_bias_grid(y,make_oof(42)['winner_50_50']).head())
else:print('Hierarchical and threshold diagnostics available; optionally enable RUN_EXTRA_DIAGNOSTICS.')

## 11. Unsupervised exploratory work — PCA, not a finalist

PCA was screened historically as a feature transformation; it is an unsupervised projection and does not require class labels to fit. It did **not** produce a selected supervised submission model. No evidence supports claiming that a standalone clustering model was trained or submitted. PCA here is **EDA only**, and is trained on the labeled training features, not test labels.

In [ ]:
Z=StandardScaler().fit_transform(unique[FEATURES]);pca=PCA(n_components=2).fit(Z)
coords=pca.transform(Z)
fig,ax=plt.subplots(figsize=(6.2,4))
for c in classes:ax.scatter(coords[y==c,0],coords[y==c,1],s=6,alpha=.25,label=f'class {c}')
ax.set(xlabel='PC1',ylabel='PC2',title='Unsupervised PCA projection (EDA only)');ax.legend();plt.tight_layout();plt.show()
print('Variance explained by PC1 and PC2:',pca.explained_variance_ratio_.round(4))

## 12. New supplemental experiment: 20% of the available training partition

**This check was performed while compiling the final report, not in the original notebook rounds.** A fixed stratified 80/20 train/validation split yields 6,000 training and 1,500 validation records. A nested 20% subset of that 6,000-row training partition has 1,200 records. Both variants are evaluated on the same fixed validation partition. This illustration is **not** an untouched holdout for the entire project because previous experiments have used these underlying records.

In [ ]:
tr,va=train_test_split(np.arange(len(y)),test_size=.20,random_state=601,stratify=y)
small,_=train_test_split(tr,train_size=.20,random_state=602,stratify=y[tr])
learning=[]
for lbl,idx in [('20% of training (1,200)',small),('All training (6,000)',tr)]:
    pA=model_for('dropcity').fit(FEATURE_SETS['dropcity'].iloc[idx],y[idx]).predict_proba(FEATURE_SETS['dropcity'].iloc[va])
    pB=model_for('elastic').fit(FEATURE_SETS['elastic'].iloc[idx],y[idx]).predict_proba(FEATURE_SETS['elastic'].iloc[va])
    yh=(.5*pA+.5*pB).argmax(1)
    learning.append({'training regime':lbl,'train rows':len(idx),'validation rows':len(va),'accuracy':accuracy_score(y[va],yh),'macro F1':f1_score(y[va],yh,average='macro')})
print(pd.DataFrame(learning).to_string(index=False))

## 13. Final top five: fit on all 7,500 distinct labeled records and export Kaggle CSVs

**Important:** This cell implements the *exact model formulations* used for the five historical Kaggle CSVs, then compares outputs to the old CSV if they happen to be present. The new output files use prefix `top0N_` so they are clearly the selected final collection. These comparisons were checked against all five original CSVs prior to publication: **0 mismatched predicted labels for every finalist** in the reference environment.

The competition evaluates submissions by `vehicle_id`, `vehicle_class` with 2,500 rows. We preserve the sample-submission order. We do not fit on unknown private labels or on public leaderboard outcomes.

In [ ]:
OUTPUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/content') if Path('/content').is_dir() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
probas={}
for name in ('baseline','dropcity','elastic','engineered'):
    m=model_for(name).fit(FEATURE_SETS[name],y)
    probas[name]=m.predict_proba(TEST_SETS[name])
probas['winner_50_50']=.5*probas['dropcity']+.5*probas['elastic']
lda_model=make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage=0))
probas['lda20']=.4*probas['dropcity']+.4*probas['elastic']+.2*lda_model.fit(unique[FEATURES],y).predict_proba(test[FEATURES])
finalists=[
 ('top01_ensemble_905.csv','winner_50_50','round4_blend_baseline_elastic.csv',.905),
 ('top02_dropcity_904.csv','dropcity','vehicle_submission_drop_city.csv',.904),
 ('top03_elastic_904.csv','elastic','round3_elastic_full_5.csv',.904),
 ('top04_engineered_903.csv','engineered','vehicle_submission_engineered.csv',.903),
 ('top05_lda20_902.csv','lda20','round5_lda20.csv',.902),
]
submission_audit=[]
for newfile,name,historical,public_score in finalists:
    preds=classes[probas[name].argmax(axis=1)]
    df=sample[[ID]].copy();df[TARGET]=preds
    assert len(df)==len(test) and df[ID].equals(sample[ID])
    assert set(df[TARGET]).issubset(set(classes))
    df.to_csv(OUTPUT_DIR/newfile,index=False)
    history=Path(files[FILES[0]]).parent/historical
    if not history.exists():history=Path('/mnt/data')/historical
    diffs='Not provided'
    if history.is_file():
        prev=pd.read_csv(history)
        assert df[ID].equals(prev[ID])
        diffs=int(np.count_nonzero(df[TARGET].to_numpy()!=prev[TARGET].to_numpy()))
    submission_audit.append({'final_file':newfile,'historical_file':historical,'confirmed_public_accuracy':public_score,'differences_if_historical_available':diffs})
print(pd.DataFrame(submission_audit).to_string(index=False))
# Standalone L2 baseline is preserved separately as a reserve, not one of the chosen five.
reserve=sample[[ID]].copy();reserve[TARGET]=classes[probas['baseline'].argmax(1)]
reserve.to_csv(OUTPUT_DIR/'reserve_baseline_902.csv',index=False)
print('Saved five finalists plus one reserve under',OUTPUT_DIR)

## 14. Historical experiment log — original notebook and Rounds 1–8

| Phase | Main syllabus-relevant hypotheses | Result / decision |
|---|---|---|
| Original Colab | Ridge/Lasso least squares, GaussianNB, LDA, QDA, L1/L2 softmax; augmentation and noisy validation | Augmented logistic public **89.60%**; noisy-validation objective not aligned with leaderboard |
| Data-optimized baseline | Clean fold validation; 8 mechanical features; augmented blends | Baseline **90.20%**, engineered **90.30%**, drop-city **90.40%** |
| Round 2 | Stronger L2, LDA blend | **89.80–90.10%** public; none improved drop-city |
| Round 3 | L1/Elastic Net, remove vehicle-volume features | Elastic Net **90.40%**; minus-volume **90.00%** |
| Round 4 | Ensemble baseline + Elastic Net; probability class bias | Winner **90.50%**, bias candidate **90.00%** |
| Round 5 | Add LDA 20%, adjust blend ratio | LDA20 **90.20%**, 75/25 blend **90.30%** |
| Round 6 | Extra Trees blended into linear models | 10% / 20% tree blends both **90.00%** |
| Round 7 | Class-0-focused power/size polynomial/interaction terms | focused **90.20%**, power-poly **90.10%** |
| Round 8 | Cubic terms, class-0 hard-example weights, error audit | No reliable gain; class-0 confusion remains major error source |

Additional screened approaches included binary class formulations, QDA/LDA covariance variants, regularized least squares, class biases, selective ablations and PCA-based representation variants. Outside-syllabus algorithms explored historically are excluded from the official selectable models.

## 15. Decision analysis and private-test risk

- Public scores of 90.50%, 90.40%, 90.40%, 90.30% and 90.20% were **observed** for the five submitted models. Their private-test scores are **unknown**. No model can be declared the best private performer without private labels.
- On three repeated five-fold OOF partitions of distinct training features, the simpler drop-city model had mean accuracy of about **89.36%**, while the winner blend had about **89.35%**. This minuscule difference is not conclusive, but motivates retaining both.
- The error audit found **772 of 792** five-fold validation mistakes involved class 0. Targeting class-0 recall alone traded off class-1/2 errors rather than improving overall accuracy.
- A 91% public/private result was **not** achieved in these experiments. Optimizing too closely to public feedback risks selecting noise. Choose the five diverse documented submissions rather than five small variants of the winning blend.
- The 20% learning check supports the interpretation that the available training data may still be informative, but the CV and leaderboard evidence does not justify a promise of 91%.

## 16. Conclusions and reproducibility checklist

**Conclusion:** A syllabus-compliant ensemble of multinomial Logistic Regression and Elastic Net achieves the highest known public score **0.90500**. Four selected companion submissions provide alternatives involving feature selection, regularization, feature engineering, and LDA blending. The project demonstrates fold-aware preprocessing, deduplication, numeric feature engineering, model-family comparisons, repeated stratified CV, error audit, class-specific decision analysis, and ethical separation of public vs private evaluation.

**Checklist:** [x] Original ML notebook methodology included (without secrets); [x] all 20 predictors documented; [x] no categorical columns falsely encoded; [x] 2,500 duplicates removed before CV; [x] scaler fitted within folds; [x] five historical top submissions exactly reproduced in audit; [x] feature engineering/negative results recorded; [x] added 20% training comparison clearly labeled as new; [x] no private-test accuracy claim.

## Appendix A. Sanitized original `Vehicle_Classification.ipynb` ML cells

The code below preserves the originally uploaded notebook's course-related content **as historical reference**, without activating duplicate state during Run All. Repository/git/auth cells were excluded because they are unrelated to the experiment and contained a hardcoded credential. One typographical error was corrected (`t0.1` → `0.1`) and the Colab upload/download steps were made safe. The executable model comparison above supersedes the original noisy-validation selection logic.

### Original ML cell 2 (historical code, not auto-executed)

```python
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes import GaussianNB
from sklearn.base import BaseEstimator, ClassifierMixin
SEED = 42
```

### Original ML cell 3 (historical code, not auto-executed)

```python
import os
if not os.path.exists("vehicle_train_10000.csv"):
    print("Place the three CSVs in the runtime and rerun this cell.")
train = pd.read_csv("vehicle_train_10000.csv")
test  = pd.read_csv("vehicle_test.csv")
sub   = pd.read_csv("vehicle_sample_submission.csv")
print(train.shape, test.shape, sub.shape)
print(train.columns.tolist())
train.info()
train.head()
```

### Original ML cell 4 (historical code, not auto-executed)

```python
FEATURES = [c for c in train.columns if c not in ("vehicle_id", "vehicle_class")]
print("duplicate rows (ignoring id):", train.duplicated(subset=FEATURES).sum())
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)
X, y = df[FEATURES].values, df["vehicle_class"].values
X_test = test[FEATURES].values
print("after dedup:", X.shape, "| class balance:", np.bincount(y) / len(y))
print("missing values:", df.isna().sum().sum(), test.isna().sum().sum())
```

### Original ML cell 5 (historical code, not auto-executed)

```python
class RegularizedLeastSquares(BaseEstimator, ClassifierMixin):
    """Ridge / Lasso regression on one-hot targets, predict argmax."""
    def __init__(self, kind="ridge", alpha=1.0): self.kind, self.alpha = kind, alpha
    def fit(self, X, y):
        self.classes_ = np.unique(y); Y = (y[:, None] == self.classes_).astype(float)
        reg = Ridge(alpha=self.alpha) if self.kind == "ridge" else Lasso(alpha=self.alpha)
        self.m_ = reg.fit(X, Y); return self
    def predict(self, X): return self.classes_[self.m_.predict(X).argmax(1)]

S = lambda m: make_pipeline(StandardScaler(), m)
CANDIDATES = {
    "Ridge least squares":   lambda: S(RegularizedLeastSquares("ridge", 10)),
    "Lasso least squares":   lambda: S(RegularizedLeastSquares("lasso", 0.01)),
    "Gaussian Naive Bayes":  lambda: S(GaussianNB()),
    "LDA":                   lambda: S(LDA()),
    "QDA (reg=0.1)":         lambda: S(QDA(reg_param=0.1)),
    "QDA (reg=0.3)":         lambda: S(QDA(reg_param=0.3)),
    "Softmax L1 (C=0.1)":    lambda: S(LogisticRegression(penalty="l1", solver="saga", C=0.1, max_iter=5000)),
    "Softmax L2 (C=1)":      lambda: S(LogisticRegression(C=1, max_iter=3000)),
    "Softmax L2 (C=0.1)":    lambda: S(LogisticRegression(C=0.1, max_iter=3000)),
    "Softmax L2 (C=0.01)":   lambda: S(LogisticRegression(C=0.01, max_iter=3000)),
}
NOISE_LEVELS = [0, 0.15, 0.3, 0.5]   # noise sd as a fraction of each feature's sd
sd = X.std(0)

def fit_with_aug(model, Xtr, ytr, aug, rng, copies=5):
    """Train on clean rows + noisy copies (aug=0 -> plain training)."""
    if aug > 0:
        Xtr = np.vstack([Xtr] + [Xtr + rng.normal(size=Xtr.shape) * sd * aug for _ in range(copies)])
        ytr = np.tile(ytr, copies + 1)
    return model.fit(Xtr, ytr)

def cv_scores(make_model, aug=0.0, n_splits=5, seed=SEED):
    rng = np.random.default_rng(seed); acc = np.zeros(len(NOISE_LEVELS))
    for tr, va in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(X, y):
        m = fit_with_aug(make_model(), X[tr], y[tr], aug, rng)
        for j, k in enumerate(NOISE_LEVELS):
            acc[j] += (m.predict(X[va] + rng.normal(size=X[va].shape) * sd * k) == y[va]).mean() / n_splits
    return acc
```

### Original ML cell 6 (historical code, not auto-executed)

```python
rows = {name: cv_scores(mk) for name, mk in CANDIDATES.items()}
res = pd.DataFrame(rows, index=[f"val_noise={k}" for k in NOISE_LEVELS]).T
res["mean_over_noise"] = res.mean(axis=1)
pd.set_option("display.width", 200)
print(res.sort_values("mean_over_noise", ascending=False).round(4))
```

### Original ML cell 7 (historical code, not auto-executed)

```python
grid = []
for C in [0.03, 0.1, 0.3, 1]:
    for aug in [0, 0.15, 0.3, 0.5]:
        a = cv_scores(lambda C=C: S(LogisticRegression(C=C, max_iter=3000)), aug=aug)
        grid.append([C, aug, *a, a.mean()])
tune = pd.DataFrame(grid, columns=["C", "train_noise"] + [f"val_noise={k}" for k in NOISE_LEVELS] + ["mean"])
print(tune.sort_values("mean", ascending=False).round(4).head(8))
best = tune.sort_values("mean", ascending=False).iloc[0]
BEST_C, BEST_AUG = float(best["C"]), float(best["train_noise"])
print("chosen C, train_noise:", BEST_C, BEST_AUG)   # expected: 0.1, 0.3
```

### Original ML cell 8 (historical code, not auto-executed)

```python
final = fit_with_aug(S(LogisticRegression(C=BEST_C, max_iter=3000)), X, y, BEST_AUG, np.random.default_rng(SEED))
pred = final.predict(X_test)
submission = pd.DataFrame({"vehicle_id": test["vehicle_id"], "vehicle_class": pred})
assert list(submission.vehicle_id) == list(sub.vehicle_id)
submission.to_csv("submission.csv", index=False)
print(submission.vehicle_class.value_counts(normalize=True).round(3))
try:
    print("Use the Colab Files pane to download the submission.")
except ImportError:
    pass
```

## Appendix B. Reproduction and source map

Original sanitized methodology: Appendix A; early clean CV and engineered features: Sections 5–9; Rounds 2–3 regularization, LDA and ablations: Sections 7, 10, 14; Round 4 best ensemble and bias experiment: Sections 8, 10, 13; Round 5 alternative ensemble ratios: Section 14; Round 6 trees: Sections 7 and 14; Round 7 focused polynomial features and holdout: Sections 5, 10, 14; Round 8 class-0 error audit: Sections 8, 10, 15. Use the accompanying archive for the per-round original files and raw historical validation CSVs.

For exact private leaderboard evaluation, submit the five CSV files from Section 13 independently; the competition organizer alone can evaluate the private labels.